In [50]:
import pandas as pd
import json
import re

#### Read in Exposure, Occ Details, LMI (wage + growth)

In [51]:
# # TEMP: Read in exposure file until full composite exposure in developed
# pca = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/TBL-Data-AI-Exposure-What-do-we-know-202602-Updated.xlsx", sheet_name="FA1", skiprows=6)
# pca = pca[["SOC2018.1",  "PCA Weighted Score.1",  "Z Score Variance.1"]]
# pca.head()

# # Requires: "title", "soc", "summary"
# onet_summ = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Occupation Data.xlsx")
# onet_summ["O*NET-SOC Code"] = onet_summ["O*NET-SOC Code"].apply(lambda x: re.sub(r'\.\d+$', '', x))

# # Requires: "medianWage", "employment" (may need to pull in Lightcast to backfill missing)
# wage = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/OEWS/Arizona 2024 OEWS.xlsx", skiprows=1, sheet_name="Annual")
# wage = wage[["SOC Code1", "50th Percentile Wage", "Rounded Employment"]]

# # Requires: "annualOpenings", "projectedGrowth", "typicalEducation"
# lmi = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/AI PEW 08182025.xlsx")

# lmi = lmi[["SOC Code", "Annual Total Openings", "Annual Percent Change", "EducationValue", "WorkExperienceValue", "GrowthRate"]]


from pathlib import Path

BASE = Path(r"C:\Users\301533\Documents\GitHub\AI\ai_impact_analysis\ai_skills")

# TEMP: Read in exposure file until full composite exposure is developed
pca = pd.read_excel(
    BASE / "TBL-Data-AI-Exposure-What-do-we-know-202602-Updated.xlsx",
    sheet_name="FA1",
    skiprows=6
)
pca = pca[["SOC2018.1", "PCA Weighted Score.1", "Z Score Variance.1"]]
pca.head()

# Requires: "title", "soc", "summary"
onet_summ = pd.read_excel(
    BASE / "ONET_30.2_Database" / "Occupation Data.xlsx"
)
onet_summ["O*NET-SOC Code"] = onet_summ["O*NET-SOC Code"].apply(
    lambda x: re.sub(r'\.\d+$', '', x)
)

# Requires: "medianWage", "employment"
wage = pd.read_excel(
    BASE / "OEWS" / "Arizona 2024 OEWS.xlsx",
    skiprows=1,
    sheet_name="Annual"
)
wage = wage[["SOC Code1", "50th Percentile Wage", "Rounded Employment"]]

# Requires: "annualOpenings", "projectedGrowth", "typicalEducation"
lmi = pd.read_excel(
    BASE / "AI PEW 08182025.xlsx"
)
lmi = lmi[[
    "SOC Code",
    "Annual Total Openings",
    "Annual Percent Change",
    "EducationValue",
    "WorkExperienceValue",
    "GrowthRate"
]]


In [52]:
lmi.head()

,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate
0,11-1011,462,0.014890,Bachelor's degree,5 years or more,1.4890
1,11-1021,8878,0.008106,Bachelor's degree,5 years or more,0.8106
2,11-1031,39,0.009885,Bachelor's degree,Less than 5 years,0.9885
3,11-2011,8,0.005038,Bachelor's degree,Less than 5 years,0.5038
4,11-2021,468,0.008244,Bachelor's degree,5 years or more,0.8244


#### Format columns for .json for readability

In [53]:
wage['50th Percentile Wage'] = pd.to_numeric(wage['50th Percentile Wage'], errors='coerce')
wage['50th Percentile Wage'] = wage['50th Percentile Wage'].map('${:,.0f}'.format)

wage['Rounded Employment'] = pd.to_numeric(wage['Rounded Employment'], errors='coerce').fillna(0).astype(int)
wage['Rounded Employment'] = wage['Rounded Employment'].map('{:,d}'.format)

lmi['Annual Total Openings'] = pd.to_numeric(lmi['Annual Total Openings'], errors='coerce').fillna(0).astype(int)
lmi['Annual Total Openings'] = lmi['Annual Total Openings'].map('{:,d}'.format)

lmi["Annual Percent Change"] = pd.to_numeric(lmi['Annual Percent Change'], errors='coerce').fillna(0).astype(float)*100
lmi["Annual Percent Change"] = lmi["Annual Percent Change"].round(2)
lmi['Annual Percent Change'] = lmi['Annual Percent Change'].apply(lambda x: str(x) + '%')

lmi.loc[lmi['EducationValue'] == "High school diplo", 'EducationValue'] = "High school diploma"
lmi.loc[lmi['EducationValue'] == "No formal educati", 'EducationValue'] = "No formal education"
lmi.loc[lmi['EducationValue'] == "Doctoral or profe", 'EducationValue'] = "Doctoral or professional degree"
lmi.loc[lmi['EducationValue'] == "Postsecondary non", 'EducationValue'] = "Postsecondary non-degree"
lmi.loc[lmi['EducationValue'] == "Associate's degre", 'EducationValue'] = "Associate's degree"
lmi.loc[lmi['EducationValue'] == "Some college, no", 'EducationValue'] = "Some college, no degree"


In [54]:
lmi.head()

,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate
0,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890
1,11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106
2,11-1031,39,0.99%,Bachelor's degree,Less than 5 years,0.9885
3,11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038
4,11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244


#### Merge files (Note that we repeat on the 8-digit)

In [55]:
df = pd.merge(pca, onet_summ, left_on="SOC2018.1", right_on="O*NET-SOC Code", how="inner")
df = pd.merge(df, lmi, left_on="SOC2018.1", right_on="SOC Code", how="inner")
df = pd.merge(df, wage, left_on="SOC2018.1", right_on="SOC Code1", how="left")

# Create "exposure" based on PCA Weighted Score.1 quintiles:
df["exposure"] = pd.qcut(df["PCA Weighted Score.1"], q=5, labels=["Very Low", "Low", "Medium", "High", "Very High"])


In [56]:
df.head()

,SOC2018.1,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code,Title,Description,SOC Code,Annual Total Openings,Annual Percent Change,EducationValue,WorkExperienceValue,GrowthRate,SOC Code1,50th Percentile Wage,Rounded Employment,exposure
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High


In [57]:
# # Requires "relatedOccupationIds"
# related = pd.read_excel("/Users/ferrisramadan/Desktop/ai_analysis/ai_impact_analysis/ai_skills/ONET_30.2_Database/Related Occupations.xlsx")


# Requires "relatedOccupationIds"
related = pd.read_excel(
    BASE / "ONET_30.2_Database" / "Related Occupations.xlsx"
)

# Filter out records not ending in "00" in SOC Code1
related = related[related["O*NET-SOC Code"].astype(str).str.endswith("00")]

# Drop last 3 characters from O*NET-SOC Code
related["O*NET-SOC Code"] = related["O*NET-SOC Code"].str[:-3]
related["Related O*NET-SOC Code"] = related["Related O*NET-SOC Code"].str[:-3]

# Merge in "PCA Weighted Score.1" from df and filter out "Very High" and "High" exposure occupations
related = pd.merge(related, df[["SOC2018.1", "exposure"]], left_on="O*NET-SOC Code", right_on="SOC2018.1", how="left")
related = related[related["exposure"].isin(["Very High", "High"])]

# Merge in "PCA Weighted Score.1" from df for related occupations
related = pd.merge(
    related,
    df[["SOC2018.1", "exposure"]].rename(columns={"exposure": "related_exposure", "SOC2018.1": "SOC_related"}),
    left_on="Related O*NET-SOC Code",
    right_on="SOC_related",
    how="left"
)

# Order related_exposure by "Low", "Medium", "High", for each "O*NET-SOC Code" group
rank_map = {
    "Very Low": 1,
    "Low": 2,
    "Medium": 3,
    "High": 4,
    "Very High": 5
}

related["exposure_rank"] = related["related_exposure"].map(rank_map)

related = related.sort_values(
    by=["Title", "exposure_rank"],
    ascending=[True, True]
)

# Keep unique O*NET-SOC Code and Related O*NET-SOC Code retain all variables
related = related.drop_duplicates(subset=["O*NET-SOC Code", "Related O*NET-SOC Code"])

In [58]:
related

,O*NET-SOC Code,Title,Related O*NET-SOC Code,Related Title,Relatedness Tier,Index,SOC2018.1,exposure,SOC_related,related_exposure,exposure_rank
1992,13-2011,Accountants and Auditors,13-1111,Management Analysts,Supplemental,12,13-2011,Very High,13-1111,Medium,3
1978,13-2011,Accountants and Auditors,13-2061,Financial Examiners,Primary-Short,2,13-2011,Very High,13-2061,High,4
1979,13-2011,Accountants and Auditors,11-3031,Treasurers and Controllers,Primary-Short,3,13-2011,Very High,11-3031,High,4
1986,13-2011,Accountants and Auditors,13-2031,Budget Analysts,Primary-Long,6,13-2011,Very High,13-2031,High,4
1988,13-2011,Accountants and Auditors,13-2052,Personal Financial Advisors,Primary-Long,8,13-2011,Very High,13-2052,High,4
...,...,...,...,...,...,...,...,...,...,...,...
5407,19-1023,Zoologists and Wildlife Biologists,25-1053,"Environmental Science Teachers, Postsecondary",Supplemental,16,19-1023,High,25-1053,High,4
5408,19-1023,Zoologists and Wildlife Biologists,25-1041,"Agricultural Sciences Teachers, Postsecondary",Supplemental,17,19-1023,High,25-1041,High,4
5410,19-1023,Zoologists and Wildlife Biologists,19-2043,Hydrologists,Supplemental,19,19-1023,High,19-2043,High,4
5375,19-1023,Zoologists and Wildlife Biologists,19-1029,Biologists,Primary-Short,1,19-1023,High,19-1029,Very High,5


In [59]:
related = related.sort_values(
    ["O*NET-SOC Code", "related_exposure"],
    ascending=[True, True]
).copy()

# 2) Create a fresh per-group order based on the CURRENT row order
related["rel_num"] = related.groupby("O*NET-SOC Code").cumcount() + 1

# 3) Pivot wider using the fresh sequence
related_pivot = related.pivot(
    index="O*NET-SOC Code",
    columns="rel_num",
    values=["Related Title", "Related O*NET-SOC Code"]
)

# 4) Flatten columns
related_pivot.columns = [
    f"{col1}_{col2}" for col1, col2 in related_pivot.columns
]

# Turn O*NET-SOC Code back into a regular column
related_pivot = related_pivot.reset_index()

# Merge GrowthRate
related_pivot = related_pivot.merge(
    lmi[["SOC Code", "GrowthRate"]],
    left_on="O*NET-SOC Code",
    right_on="SOC Code",
    how="left"
)

# Flag above-average growth
related_pivot["GrowthRateAboveAvg"] = (
    related_pivot["GrowthRate"] > related_pivot["GrowthRate"].mean()
)

In [60]:
related_pivot

,O*NET-SOC Code,Related Title_1,Related Title_2,Related Title_3,Related Title_4,Related Title_5,Related Title_6,Related Title_7,Related Title_8,Related Title_9,...,Related O*NET-SOC Code_14,Related O*NET-SOC Code_15,Related O*NET-SOC Code_16,Related O*NET-SOC Code_17,Related O*NET-SOC Code_18,Related O*NET-SOC Code_19,Related O*NET-SOC Code_20,SOC Code,GrowthRate,GrowthRateAboveAvg
0,11-1011,Management Analysts,Social and Community Service Managers,Administrative Services Managers,General and Operations Managers,Treasurers and Controllers,Compliance Managers,"Education Administrators, Postsecondary",Chief Sustainability Officers,First-Line Supervisors of Office and Administr...,...,43-6011,41-1012,27-3031,13-1075,43-6014,11-1031,NaN,11-1011,1.4890,True
1,11-1021,First-Line Supervisors of Housekeeping and Jan...,Administrative Services Managers,Industrial Production Managers,Facilities Managers,First-Line Supervisors of Entertainment and Re...,"First-Line Supervisors of Mechanics, Installer...",First-Line Supervisors of Production and Opera...,Management Analysts,First-Line Supervisors of Construction Trades ...,...,43-1011,41-1012,13-1082,15-1299,17-2112,53-1042,53-1043,11-1021,0.8106,False
2,11-2011,Management Analysts,Merchandise Displayers and Window Trimmers,Marketing Managers,Market Research Analysts and Marketing Special...,Sales Managers,Fundraisers,Art Directors,Advertising Sales Agents,Public Relations Specialists,...,41-3091,13-1011,11-3061,41-9041,15-1299,15-2051,NaN,11-2011,0.5038,False
3,11-2021,Management Analysts,Market Research Analysts and Marketing Special...,Sales Managers,Advertising and Promotions Managers,Financial and Investment Analysts,General and Operations Managers,Advertising Sales Agents,Public Relations Managers,Public Relations Specialists,...,41-3031,13-1082,13-1081,27-3043,15-2051,13-1022,NaN,11-2021,0.8244,True
4,11-2022,Management Analysts,Marketing Managers,Market Research Analysts and Marketing Special...,Advertising and Promotions Managers,First-Line Supervisors of Retail Sales Workers,General and Operations Managers,Chief Executives,Financial and Investment Analysts,First-Line Supervisors of Office and Administr...,...,13-1199,43-4051,41-3031,13-1081,43-4011,13-1022,13-1023,11-2022,0.4360,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,43-9081,"Office Machine Operators, Except Computer",Archivists,Editors,Desktop Publishers,Technical Writers,Word Processors and Typists,File Clerks,Writers and Authors,Data Entry Keyers,...,27-3092,43-9111,25-4022,19-4061,31-9094,43-4021,NaN,43-9081,-1.7393,False
246,43-9111,Management Analysts,Statisticians,"Bookkeeping, Accounting, and Auditing Clerks",Document Management Specialists,Health Information Technologists and Medical R...,Social Science Research Assistants,Bioinformatics Technicians,Database Architects,File Clerks,...,15-1211,15-2031,15-1242,15-2051,NaN,NaN,NaN,43-9111,-0.3945,False
247,51-6092,"Sewers, Hand","Tailors, Dressmakers, and Custom Sewers","Cutters and Trimmers, Hand",Sewing Machine Operators,"Molders, Shapers, and Casters, Except Metal an...",Shoe and Leather Workers and Repairers,"Painting, Coating, and Decorating Workers",Shoe Machine Operators and Tenders,"Patternmakers, Metal and Plastic",...,27-1012,51-4111,51-4061,51-6062,27-1022,27-1021,NaN,51-6092,0.0000,False
248,51-9162,"Multiple Machine Tool Setters, Operators, and ...",Engine and Other Machine Assemblers,Computer Numerically Controlled Tool Operators,Machinists,"Model Makers, Metal and Plastic","Patternmakers, Metal and Plastic","Milling and Planing Machine Setters, Operators...","Lathe and Turning Machine Tool Setters, Operat...",Robotics Technicians,...,17-2199,17-2061,15-1251,51-2023,51-2022,NaN,NaN,51-9162,1.5191,True


In [61]:
def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', str(title).lower()).strip('-')

cols = [c for c in related_pivot.columns if c.startswith("Related Title_")]

if not cols:
    raise ValueError("No columns found starting with 'Related Title_'")

related_pivot["related_titles_clean"] = related_pivot[cols].apply(
    lambda row: ", ".join(
        f'"{make_id(val)}"'
        for val in row
        if pd.notna(val) and str(val).strip() != ""
    ),
    axis=1
)

In [62]:
df2 = pd.merge(df, related_pivot[["O*NET-SOC Code", "GrowthRateAboveAvg", "related_titles_clean"]], left_on="SOC2018.1", right_on="O*NET-SOC Code", how="left")


In [63]:
# Clean up names
df2.rename(columns={"Title": "title",
                   'SOC2018.1':"soc", 
                   "Description": "summary",
                   "50th Percentile Wage": "medianWage",
                   "Annual Total Openings": "annualOpenings",
                   "Rounded Employment": "employment", 
                   "Annual Percent Change": "projectedGrowth",
                   "EducationValue": "typicalEducation"
                   }, inplace=True)

In [64]:
df2.head()

,soc,PCA Weighted Score.1,Z Score Variance.1,O*NET-SOC Code_x,title,summary,SOC Code,annualOpenings,projectedGrowth,typicalEducation,WorkExperienceValue,GrowthRate,SOC Code1,medianWage,employment,exposure,O*NET-SOC Code_y,GrowthRateAboveAvg,related_titles_clean
0,11-1011,0.818306,0.395745,11-1011,Chief Executives,Determine and formulate policies and provide o...,11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""management-analysts"", ""social-and-community-s..."
1,11-1011,0.818306,0.395745,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",11-1011,462,1.49%,Bachelor's degree,5 years or more,1.4890,11-1011,"$150,590","3,100",High,11-1011,True,"""management-analysts"", ""social-and-community-s..."
2,11-1021,0.696313,0.213237,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",11-1021,"8,878",0.81%,Bachelor's degree,5 years or more,0.8106,11-1021,"$90,000","100,340",High,11-1021,False,"""first-line-supervisors-of-housekeeping-and-ja..."
3,11-2011,0.678133,0.185190,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",11-2011,8,0.5%,Bachelor's degree,Less than 5 years,0.5038,11-2011,"$107,000",50,High,11-2011,False,"""management-analysts"", ""merchandise-displayers..."
4,11-2021,0.783175,0.237888,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",11-2021,468,0.82%,Bachelor's degree,5 years or more,0.8244,11-2021,"$135,920","4,350",High,11-2021,True,"""management-analysts"", ""market-research-analys..."


In [65]:
def parse_related(val):
    if pd.isna(val) or str(val).strip() == "":
        return []
    
    return [
        x.strip().strip('"')
        for x in str(val).split(",")
        if x.strip() != ""
    ]

def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

grouped = df2.groupby([
    "title", "soc", "exposure", "summary",
    "medianWage", "annualOpenings",
    "employment", "projectedGrowth",
    "typicalEducation"
], dropna=False)

output = []

for keys, group in grouped:
    (
        title, soc, exposure, summary,
        medianWage, annualOpenings,
        employment, projectedGrowth,
        typicalEducation
    ) = keys

    training = []
    for _, row in group.iterrows():
        if pd.notna(row.get("provider")):
            training.append({
                "provider": row.get("provider"),
                "program": row.get("program"),
                "cip": row.get("cip"),
                "award": row.get("award"),
                "location": row.get("location")
            })

    related_ids = (
        parse_related(group["related_titles_clean"].dropna().iloc[0])
        if group["related_titles_clean"].notna().any()
        else []
    )

    obj = {
        "id": make_id(title),
        "title": title,
        "soc": soc,
        "exposure": exposure,
        "summary": summary,
        "laborMarket": {
            "medianWage": str(medianWage),
            "annualOpenings": str(annualOpenings),
            "employment": str(employment),
            "projectedGrowth": str(projectedGrowth),
            "typicalEducation": typicalEducation
        },
        "relatedOccupationIds": related_ids,
        "training": training
    }

    output.append(obj)

with open("data.json", "w") as f:
    json.dump(output, f, indent=2)

C:\Users\301533\AppData\Local\Temp\ipykernel_31084\863912508.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df2.groupby([


In [66]:
import pandas as pd
import json
from pathlib import Path

# =====================================================
# BUILD FERRIS OCCUPATION MASTER DATA
# =====================================================

def make_id(title):
    return re.sub(r'[^a-z0-9]+', '-', str(title).lower()).strip('-')

ferris_occ = df[[
    "SOC2018.1",
    "Title",
    "Description",
    "exposure",
    "50th Percentile Wage",
    "Annual Total Openings",
    "Rounded Employment",
    "Annual Percent Change",
    "EducationValue",
    "WorkExperienceValue",
    "GrowthRate"
]].copy()

ferris_occ = ferris_occ.rename(columns={
    "SOC2018.1": "soc",
    "Title": "title",
    "Description": "summary",
    "50th Percentile Wage": "medianWage",
    "Annual Total Openings": "annualOpenings",
    "Rounded Employment": "employment",
    "Annual Percent Change": "projectedGrowth",
    "EducationValue": "typicalEducation",
    "WorkExperienceValue": "workExperience",
    "GrowthRate": "growthRate"
})


ferris_occ["id"] = ferris_occ["title"].apply(make_id)

ferris_occ.head()

,soc,title,summary,exposure,medianWage,annualOpenings,employment,projectedGrowth,typicalEducation,workExperience,growthRate,id
0,11-1011,Chief Executives,Determine and formulate policies and provide o...,High,"$150,590",462,"3,100",1.49%,Bachelor's degree,5 years or more,1.4890,chief-executives
1,11-1011,Chief Sustainability Officers,"Communicate and coordinate with management, sh...",High,"$150,590",462,"3,100",1.49%,Bachelor's degree,5 years or more,1.4890,chief-sustainability-officers
2,11-1021,General and Operations Managers,"Plan, direct, or coordinate the operations of ...",High,"$90,000","8,878","100,340",0.81%,Bachelor's degree,5 years or more,0.8106,general-and-operations-managers
3,11-2011,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici...",High,"$107,000",8,50,0.5%,Bachelor's degree,Less than 5 years,0.5038,advertising-and-promotions-managers
4,11-2021,Marketing Managers,"Plan, direct, or coordinate marketing policies...",High,"$135,920",468,"4,350",0.82%,Bachelor's degree,5 years or more,0.8244,marketing-managers


In [67]:
# =====================================================
# LOAD ARIELLE SANKEY / RELATED OCCUPATION FILE
# =====================================================

sankey_file = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\High_VeryHigh_AND_Medium_Related_Occs_Final.xlsx"
)

sankey = pd.read_excel(sankey_file)

sankey.head()

,Pathway Type,Source SOC,Source Occupation,Source AI Exposure,Gateway Rank,Gateway SOC,Gateway Occupation,Gateway AI Exposure,Related Occupation SOC,Related Occupation,...,Related Occupation O*NET Rank,Average Source Rank,Combined Rank,Lightcast Compatibility Index,Related Occupation O*NET Tier,Related Occupation O*NET Index,Lightcast Score,Related Occupation O*NET Score,Combined Lightcast + O*NET Score,Combined Score 0-100
0,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-9151,Social and Community Service Managers,...,3.0,3.0,3,NaN,Primary-Short,2.0,NaN,0.977778,0.977778,97.8
1,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-1031,Legislators,...,11.0,11.0,15,NaN,Primary-Long,9.0,NaN,0.678889,0.678889,67.9
2,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,13-1111,Management Analysts,...,14.0,14.0,20,NaN,Supplemental,11.0,NaN,0.400556,0.400556,40.1
3,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-3012,Administrative Services Managers,...,15.0,15.0,21,NaN,Supplemental,12.0,NaN,0.389444,0.389444,38.9
4,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,39-1014,First-Line Supervisors of Entertainment and Re...,...,28.0,28.0,33,NaN,Supplemental,20.0,NaN,0.245000,0.245000,24.5


In [68]:
sankey.columns.tolist()

['Pathway Type',
 'Source SOC',
 'Source Occupation',
 'Source AI Exposure',
 'Gateway Rank',
 'Gateway SOC',
 'Gateway Occupation',
 'Gateway AI Exposure',
 'Related Occupation SOC',
 'Related Occupation',
 'Related Occupation AI Exposure',
 'Relationship Source',
 'Lightcast Rank',
 'Related Occupation O*NET Rank',
 'Average Source Rank',
 'Combined Rank',
 'Lightcast Compatibility Index',
 'Related Occupation O*NET Tier',
 'Related Occupation O*NET Index',
 'Lightcast Score',
 'Related Occupation O*NET Score',
 'Combined Lightcast + O*NET Score',
 'Combined Score 0-100']

In [69]:
# =====================================================
# MERGE FERRIS OCCUPATION DATA INTO SANKEY DATA
# =====================================================

# SOURCE OCCUPATIONS
sankey2 = sankey.merge(
    ferris_occ.add_prefix("source_"),
    left_on="Source SOC",
    right_on="source_soc",
    how="left"
)

# DESTINATION OCCUPATIONS
sankey2 = sankey2.merge(
    ferris_occ.add_prefix("dest_"),
    left_on="Related Occupation SOC",
    right_on="dest_soc",
    how="left"
)

# GATEWAY OCCUPATIONS
sankey2 = sankey2.merge(
    ferris_occ.add_prefix("gateway_"),
    left_on="Gateway SOC",
    right_on="gateway_soc",
    how="left"
)

print(sankey2.shape)

sankey2.head()

(29288, 59)


,Pathway Type,Source SOC,Source Occupation,Source AI Exposure,Gateway Rank,Gateway SOC,Gateway Occupation,Gateway AI Exposure,Related Occupation SOC,Related Occupation,...,gateway_summary,gateway_exposure,gateway_medianWage,gateway_annualOpenings,gateway_employment,gateway_projectedGrowth,gateway_typicalEducation,gateway_workExperience,gateway_growthRate,gateway_id
0,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-9151,Social and Community Service Managers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-9151,Social and Community Service Managers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-1031,Legislators,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,11-1031,Legislators,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Direct,11-1011,Chief Executives,High,NaN,NaN,NaN,NaN,13-1111,Management Analysts,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
sankey2.columns.tolist()

['Pathway Type',
 'Source SOC',
 'Source Occupation',
 'Source AI Exposure',
 'Gateway Rank',
 'Gateway SOC',
 'Gateway Occupation',
 'Gateway AI Exposure',
 'Related Occupation SOC',
 'Related Occupation',
 'Related Occupation AI Exposure',
 'Relationship Source',
 'Lightcast Rank',
 'Related Occupation O*NET Rank',
 'Average Source Rank',
 'Combined Rank',
 'Lightcast Compatibility Index',
 'Related Occupation O*NET Tier',
 'Related Occupation O*NET Index',
 'Lightcast Score',
 'Related Occupation O*NET Score',
 'Combined Lightcast + O*NET Score',
 'Combined Score 0-100',
 'source_soc',
 'source_title',
 'source_summary',
 'source_exposure',
 'source_medianWage',
 'source_annualOpenings',
 'source_employment',
 'source_projectedGrowth',
 'source_typicalEducation',
 'source_workExperience',
 'source_growthRate',
 'source_id',
 'dest_soc',
 'dest_title',
 'dest_summary',
 'dest_exposure',
 'dest_medianWage',
 'dest_annualOpenings',
 'dest_employment',
 'dest_projectedGrowth',
 'dest_ty

In [71]:
sankey2[
    [
        "Source Occupation",
        "source_medianWage",
        "source_annualOpenings",
        "source_employment",
        "Related Occupation",
        "dest_medianWage",
        "dest_annualOpenings",
        "dest_employment",
        "Gateway Occupation",
        "gateway_medianWage"
    ]
].head(20)

,Source Occupation,source_medianWage,source_annualOpenings,source_employment,Related Occupation,dest_medianWage,dest_annualOpenings,dest_employment,Gateway Occupation,gateway_medianWage
0,Chief Executives,"$150,590",462,"3,100",Social and Community Service Managers,"$75,070",345,"2,870",NaN,NaN
1,Chief Executives,"$150,590",462,"3,100",Social and Community Service Managers,"$75,070",345,"2,870",NaN,NaN
2,Chief Executives,"$150,590",462,"3,100",Legislators,NaN,NaN,NaN,NaN,NaN
3,Chief Executives,"$150,590",462,"3,100",Legislators,NaN,NaN,NaN,NaN,NaN
4,Chief Executives,"$150,590",462,"3,100",Management Analysts,"$89,520","1,896","18,110",NaN,NaN
5,Chief Executives,"$150,590",462,"3,100",Management Analysts,"$89,520","1,896","18,110",NaN,NaN
6,Chief Executives,"$150,590",462,"3,100",Administrative Services Managers,"$99,800",509,"6,310",NaN,NaN
7,Chief Executives,"$150,590",462,"3,100",Administrative Services Managers,"$99,800",509,"6,310",NaN,NaN
8,Chief Executives,"$150,590",462,"3,100",First-Line Supervisors of Entertainment and Re...,"$46,120",429,"2,110",NaN,NaN
9,Chief Executives,"$150,590",462,"3,100",First-Line Supervisors of Entertainment and Re...,"$46,120",429,"2,110",NaN,NaN


In [72]:
output_path = Path(
    r"G:\.shortcut-targets-by-id\1KKrBsNyhwzbYus9ExtUkUd_3jE4VZ4W6\AI_Analysis\Arielle's Stuff\Sankey_With_Ferris_Economics_TEST.xlsx"
)

sankey2.to_excel(output_path, index=False)

In [77]:
pathways_output = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
)

sankey2.to_json(
    pathways_output,
    orient="records",
    indent=2
)

In [74]:
pathways_output = Path(
    r"C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json"
)

sankey2.to_json(
    pathways_output,
    orient="records",
    indent=2
)

print(f"Saved to: {pathways_output}")

Saved to: C:\Users\301533\Documents\GitHub\AI\AI Website\aipathways.github.io\pathways.json
